# Clustering

In [ ]:
#| eval: false

from fhemb.utils.cutils import dtw_clustering, depict_clusters, plot_dm_dendrogram, plot_clustered_dm_heatmap, \
    compute_silhouette_vs_k, plot_silhouette_curves, depict_clusters

In [ ]:
#| hide
from nbdev import show_doc

## Cluster DTW Alignments Between Pairs of Embeddings

In [ ]:
#| eval: false

show_doc(plot_clustered_dm_heatmap, name='Clustered heatmap', title_level=3)

---

### Clustered heatmap

```python

def plot_clustered_dm_heatmap(
    ts, # Arrays of shape (n_samples, n_features, n_timesteps).
    alignment_type:str='fastdtw', # Type of alignment method to use ('dcor', 'fastdtw', 'tslearn_dtw', 'soft_dtw', 'soft_dtw_div').
    gamma:int=1, # smoothing parameter:
    Small gamma → behaves more like classic DTW (hard minimum)
    Large gamma → smoother, more diffused alignment (less sensitive to exact path)
    0.01 Very close to DTW:     Precise alignment
    1.0 Balanced smoothing:     Most common default
    10.0 Very smooth: Robust to noise, good for optimization
    dist:function=euclidean, # Distance metric function to use for DTW is implicitly euclidean.
    p:int=3, # Parameter for Minkowski distance (if used).
    radius:int=1, # Radius parameter for the FastDTW algorithm.
    normalize:NoneType=None, # Normalization method to apply to the distance matrix for better visualisation.
    Options: 'max', 'minmax', 'z-score', 'l1', 'l2', 'softmax', 'sym', 'log', 'none'
    n_jobs:int=4, # number of jobs for parallel execution
    labels:NoneType=None, # Labels for time series. If None, defaults to TS0, TS1, ...
    title:str='Clustered Distance Matrix', # Title of the plot.
    color_continuous_scale:str='Viridis', # Colormap for the heatmap.
    base_size:int=20, # Base pixel size per time series (controls figure dimensions).
):


```

*Plot a clustered DTW distance matrix using hierarchical clustering and Plotly.*
The distance matrix is calculated using distance correlation (dcor) or a DTW method.

In [ ]:
#| eval: false

show_doc(plot_dm_dendrogram, name='Dendrogram', title_level=3)

---

### Dendrogram

```python

def plot_dm_dendrogram(
    ts, # Arrays of shape (n_samples, n_features, n_timesteps).
    alignment_type:str='fastdtw', # Type of alignment method to use ('dcor', 'fastdtw', 'tslearn_dtw', 'soft_dtw', 'soft_dtw_div').
    gamma:int=1, # smoothing parameter:
    Small gamma → behaves more like classic DTW (hard minimum)
    Large gamma → smoother, more diffused alignment (less sensitive to exact path)
    0.01 Very close to DTW:     Precise alignment
    1.0 Balanced smoothing:     Most common default
    10.0 Very smooth: Robust to noise, good for optimization
    dist:function=euclidean, # Distance metric function to use for DTW is implicitly euclidean.
    p:int=3, # Parameter for Minkowski distance (if used).
    radius:int=1, # Radius parameter for the FastDTW algorithm.
    n_jobs:int=4, # number of jobs for parallel execution
    labels:NoneType=None, # labels for each time series.
    method:str='average', # linkage method ('average', 'complete', 'single', etc.)
    title:str='Distance Matrix Dendrogram', # plot title.
):


```

*Plot a dendrogram from a DTW distance matrix: a hierarchical tree showing how time series cluster together based on their DTW‑ or dcor‑derived distances, with merge heights reflecting the distances at which clusters join.*
The distance matrix is calculated using distance correlation (dcor) or a DTW method.

## Cluster Dynamics

In [ ]:
#| eval: false
show_doc(compute_silhouette_vs_k, name='Compute Silhouette Scores', title_level=3)

---

### Compute Silhouette Scores

```python

def compute_silhouette_vs_k(
    ts, # Input time series data.
    interval:NoneType=None, # Optional (start, end) interval to crop each time series before clustering.
    normalize:bool=False, # Normalization method to apply. One of:
    - 'minmax'
    - 'meanvar'
    - False (no normalization)
    metric:str='dtw', # Distance metric for clustering. Supported values depend on `dtw_clustering`.
    max_k:int=10, # Maximum number of clusters to evaluate. Evaluation runs from k=2 to k=max_k.
    max_iter:int=10, # Maximum number of iterations for the clustering algorithm.
    random_state:int=0, # Random seed for reproducibility.
    n_jobs:int=-1, # Number of parallel workers. -1 uses all available cores.
): # A dictionary with the following keys:
    - 'k_values' : list[int]
        The evaluated cluster counts.
    - 'scores' : list[float]
        Silhouette scores for each k (NaN for invalid runs).
    - 'best_k' : int or None
        The k with the highest silhouette score, or None if all scores are NaN.
    - 'best_score' : float or None
        The highest silhouette score, or None if all scores are NaN.
    - 'centroids' : dict[int, np.ndarray or None]
        Mapping from k to centroid arrays (None if clustering failed).
    - 'invalid' : list[int]
        List of k-values that produced invalid or non-finite silhouette scores.


```

*Compute silhouette scores for a range of cluster counts (k) using DTW-based clustering.*

This function evaluates clustering quality across values of `k` from 2 to `max_k`
using the silhouette score returned by `dtw_clustering`. Computation is parallelized
across k-values and includes a progress bar. The function returns a canonical,
registry-style dictionary suitable for downstream plotting or analysis.

::: {.callout title="Time‑series normalization methods — API summary" collapse="true"}

| `normalize` value | Meaning / Transformation | Equation | Scope | Strengths | Notes |
|-------------------|--------------------------|----------|--------|-----------|--------|
| minmax | Per‑series min–max scaling to a fixed interval | $$ x' = \frac{x - \min(x)}{\max(x) - \min(x)} (r_{\max} - r_{\min}) + r_{\min} $$ | Per sample (each time series independently) | Preserves shape; removes amplitude differences; stable for DTW‑based methods | Defaults: $r_{\min}=0, r_{\max}=1$. If all values equal → returns zeros. |
| meanvar | Per‑series standardization to target mean and variance | $$ x' = \frac{x - \mu(x)}{\sigma(x)} \cdot \sigma_{\mathrm{target}} + \mu_{\mathrm{target}} $$ | Per sample (default) or per feature | Centers + scales each series; good for comparing shapes across samples | Defaults: $\mu_{\mathrm{target}} = 0, \sigma_{\mathrm{target}} = 1$. If per‑sample variance is 0 → returns zeros. |
:::

::: {.callout collapse="true" title="Distance metric for clustering - API summary "}
| `metric` value | Distance Function                | Barycenter Method              | Time‑Shift Invariance | Computational Cost | Recommended Use Case |
|---------------|----------------------------------|--------------------------------|------------------------|---------------------|-----------------------|
| `euclidean`   | Pointwise squared Euclidean      | Arithmetic mean                | No                     | Fast                | Aligned, equal‑length series; baseline K‑means |
| `dtw`         | Dynamic Time Warping             | DBA (DTW Barycenter Averaging) | Yes                    | Medium              | Shape‑based clustering with temporal misalignment |
| `softdtw`     | Soft‑DTW (differentiable DTW)    | Soft‑DTW barycenter            | Yes                    | Slow                | Smooth, high‑quality shape clustering |
:::

In [ ]:
#| eval: false
show_doc(plot_silhouette_curves, name='Plot Silhouette curves', title_level=3)

---

### Plot Silhouette curves

```python

def plot_silhouette_curves(
    results, # Either a single silhouette result dict, or a mapping from keys
(e.g., start times) to silhouette result dicts.
    filepath, # Output path for the combined plot.
    labels:NoneType=None, # Optional labels for each curve. If None, labels are derived from keys.
    show_best:bool=True, # Whether to draw vertical lines and annotations for each best_k.
):


```

*Plot multiple silhouette score curves on a single graph.*

In [ ]:
#| eval: false

show_doc(dtw_clustering, name='Performe DTW clustering', title_level=3)

---

### Performe DTW clustering

```python

def dtw_clustering(
    ts, # time series data.
    interval:NoneType=None, # time interval (start, end) to consider for clustering. If None, use the entire series.
    normalize:bool=False, # normalization method ('minmax', 'meanvar', or False).
    metric:str='dtw', # distance metric to use ('dtw', 'softdtw', 'gak', 'euclidean').
    n_clusters:int=3, max_iter:int=10, random_state:int=0, # random seed for reproducibility.
): # mapping cluster labels to lists of subjects.
subj_clusters: dict, mapping subjects to their assigned cluster labels.
labels: array, cluster labels for each time series.
silhouette: float or None, silhouette score if applicable.
centroids: array, cluster centroids.


```

*Perform DTW-based clustering using TimeSeriesKMeans from tslearn.*

In [ ]:
#| eval: false

show_doc(depict_clusters, name='Depict clusters', title_level=3)

---

### Depict clusters

```python

def depict_clusters(
    clusters_subjs, # Mapping cluster labels to lists of subjects.
    subj_position, # Mapping subjects to their (x, y) positions over time.
Each entry must be array-like of shape (2, T) or (T, 2).
    frame:NoneType=None, # Frame(s) to visualize. If None, defaults to [0].
):


```

*Interactive 2D scatter plot(s) of clustered subjects at given frame(s) using Plotly.*